# EMIPredict AI - Leakage-safe data preparation

This notebook prepares the 400,000-record EMI risk dataset for the next modeling stage. It deliberately performs the cleaned, stratified 70/15/15 split **before** feature engineering, statistical selection, encoding, or scaling. All learned transformations are fitted only on the training data.

The reusable implementation is in [`src/emi_preprocessing.py`](../src/emi_preprocessing.py).

In [ ]:
from pathlib import Path
import logging
import sys

import pandas as pd
+
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
+
+    from src.emi_preprocessing import (
+        CLASS_TARGET,
+        REGRESSION_TARGET,
+        DataPreprocessor,
+        FeatureEngineer,
+        StatisticalAnalyzer,
+    )
+
+    logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
+    DATA_PATH = PROJECT_ROOT / "data" / "emi_prediction_dataset.csv"
+    RANDOM_STATE = 42

## 1. Load and clean

Cleaning is deterministic and does not learn statistics. Column names are normalized, exact duplicate records are removed, blank values are treated as null, and every null in every column is replaced with the literal sentinel `missing`.

In [ ]:
preprocessor = DataPreprocessor(random_state=RANDOM_STATE)
cleaned_df = preprocessor.load_and_clean(DATA_PATH)
display(cleaned_df.head())
display(cleaned_df.dtypes.rename("dtype").to_frame())
print(f"Cleaned shape: {cleaned_df.shape}")
print(f"Missing sentinel cells: {(cleaned_df == 'missing').sum().sum():,}")

## 2. Split before learned transformations

The classification target is used only for stratification. The regression target and classification target are both excluded from model features. No imputer, encoder, scaler, feature engineering, or feature selection has run before this split.

In [ ]:
splits = preprocessor.split(cleaned_df)
for name, split in [("train", splits.train), ("validation", splits.validation), ("test", splits.test)]:
    print(f"{name:>10}: {split.shape[0]:,} rows | class proportions")
    display(split[CLASS_TARGET].value_counts(normalize=True).rename("proportion").to_frame())

## 3. Training-only statistical analysis

Pearson and Spearman tests are calculated for numeric features versus `max_monthly_emi`; chi-square tests are calculated for categorical features versus `emi_eligibility`; and ANOVA or Kruskal-Wallis is selected for numeric features versus the classification target. VIF is calculated on numeric training data to identify severe multicollinearity.

In [ ]:
analyzer = StatisticalAnalyzer(alpha=0.05)
reports = analyzer.report(splits.train)
for report_name, report in reports.items():
    print(f"\n--- {report_name} ---")
    display(report.head(15))

## 4. Domain feature engineering and transparent selection

Ratios use numeric coercion after the split, so the raw sentinel remains available for categorical values. Engineered features include total obligations, debt-to-income, expense-to-income, disposable income, liquidity months, and a bounded financial stability score. Constant columns and severe VIF violations are removed; weak univariate p-values are reported rather than blindly dropped because conditional and nonlinear predictive value cannot be ruled out by a single test.

In [ ]:
engineer = FeatureEngineer()
train_features, y_train_cls, y_train_reg = preprocessor.feature_target_frames(splits.train)
validation_features, y_validation_cls, y_validation_reg = preprocessor.feature_target_frames(splits.validation)
test_features, y_test_cls, y_test_reg = preprocessor.feature_target_frames(splits.test)
train_engineered = engineer.transform(train_features)
validation_engineered = engineer.transform(validation_features)
test_engineered = engineer.transform(test_features)
reports["vif_engineered"] = analyzer.vif_report(train_engineered)
selected_features, dropped_features = engineer.select_features(train_engineered, reports)
validation_engineered = validation_engineered[selected_features]
test_engineered = test_engineered[selected_features]
train_engineered = train_engineered[selected_features]
print(f"Selected features: {len(selected_features)}")
print(f"Dropped features (constant or VIF > 20): {dropped_features}")


## 5. Train-only preprocessing pipeline

The `ColumnTransformer` applies `RobustScaler` to numeric columns and `OneHotEncoder(handle_unknown='ignore')` to categorical columns. The pipeline is fitted only below on `train_engineered`; validation and test rows are transformed afterward without refitting.

In [ ]:
preprocessing_pipeline = engineer.build_pipeline(train_engineered)
X_train = preprocessing_pipeline.fit_transform(train_engineered)
X_validation = preprocessing_pipeline.transform(validation_engineered)
X_test = preprocessing_pipeline.transform(test_engineered)
print(f"Model-ready shapes: train={X_train.shape}, validation={X_validation.shape}, test={X_test.shape}")
print(f"Targets: classification={CLASS_TARGET}, regression={REGRESSION_TARGET}")